# Pedigree consistency and other basic properties of tandem repeats in the CEPH pedigree

In [1]:
import os
import gzip
import itertools
import pathlib
from collections import namedtuple
import importlib

import helpers.consistency as cons
import helpers.allele_db as db

importlib.reload(cons)
importlib.reload(db)

<module 'helpers.allele_db' from '/pbi/flash/edolzhenko/2024/Q2/PlatinumPedigree/pipelines/tandem-repeats/helpers/allele_db.py'>

In [2]:
pathlib.Path("output/").mkdir(exist_ok=True)

In [3]:
# Install the editdistance library
# python -m pip install editdistance

In [4]:
inheritance_vector_path = "/home/edolzhenko/flash/2024/Q2/palladium/000_ivecs.grch38.csv"

In [5]:
vcf_dir = "/pbi/flash/tmokveld/data/AWS_sync/PB_TRs/TRGT/GRCh38_v1.0_50bp_merge/493ef25/"
manifest = {
    "NA12879": os.path.join(vcf_dir, "2216_DM_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12881": os.path.join(vcf_dir, "2211_D_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12882": os.path.join(vcf_dir, "2212_S_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12883": os.path.join(vcf_dir, "2298_S_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12884": os.path.join(vcf_dir, "2215_S_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12885": os.path.join(vcf_dir, "2217_D_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12886": os.path.join(vcf_dir, "2189_SF_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12887": os.path.join(vcf_dir, "2187_D_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12877": os.path.join(vcf_dir, "2209_SF_GRCh38_50bp_merge.sorted.vcf.gz"),
    "NA12878": os.path.join(vcf_dir, "2188_DM_GRCh38_50bp_merge.sorted.vcf.gz")
}

Generate an allele database. This database is just a compressed file with columns containing repeat ids, sample ids, and allele sequences respectively.

In [6]:
db.create_allele_db(manifest, "output/allele_db.gz", sort_threads=8)

Load the inheritance vectors containing the expected inheritance patterns across (most of) the genome. 

In [7]:
ivecs = cons.load_ivecs(inheritance_vector_path)

In [8]:
def compress_homs(seq, min_len=4):
    comp_seq = ""
    for base, group in itertools.groupby(seq):
        run_len = len(list(group))
        comp_seq += base * min(min_len, run_len)
    return comp_seq


compress_homs("AAATTTTTTTTAATCGGTTTTTTT")

'AAATTTTAATCGGTTTT'

In [9]:
def compress_seqs(seqs):
    return tuple(compress_homs(s) for s in seqs)

In [ ]:
Summary = namedtuple("Summary", "trid assign num_imperfect max_error total_dist")


def summarize(hap_to_dists):
    num_imperfect = 0
    max_error = 0
    total_dist = 0
    for hap, dists in hap_to_dists.items():
        if not dists:
            continue
        if max(dists) != 0:
            num_imperfect += 1
        allele_error = (len(dists) - dists.count(0)) / len(dists)
        max_error = max(max_error, allele_error)
        total_dist += sum(dists)

    return num_imperfect, max_error, total_dist


def get_alleles(path, homopolymer_compression=False):
    with gzip.open(path, "r") as file:
        for trid, recs in itertools.groupby(file, key=lambda rec: rec.decode("utf8").split()[0]):
            if not trid.startswith("chr"):
                print(f"Skipping {trid}")
                continue
            recs = [rec.decode("utf8").split() for rec in recs]
            alleles = {sample: tuple(alleles.split(",")) for trid, sample, alleles in recs}
            if homopolymer_compression:
                alleles = {sample: compress_seqs(seqs) for sample, seqs in alleles.items()}
            yield trid, alleles


def get_ivec(inheritance_vecs, trid):
    chrom, start, end = trid.split("_")[:3]
    start, end = int(start), int(end)
    for inheritance_vec in inheritance_vecs:
        if inheritance_vec["chrom"] == chrom and inheritance_vec["start"] <= start and end <= inheritance_vec["end"]:
            return inheritance_vec
    return None


def get_score(hap_to_dists):
    score = 0
    for dists in hap_to_dists.values():
        score += sum(dists)
    return score


num_perfect = 0
num_skipped = 0
tr_summaries = []
allele_iter = get_alleles("output/allele_db.gz", homopolymer_compression=False)
for num_total, (trid, alleles) in enumerate(allele_iter):
    # Skip loci missing alleles and those with symbolic ids
    if len(alleles) != 10 or trid.count("_") != 3:
        num_skipped += 1
        continue

    ivec = get_ivec(ivecs, trid)
    # Also skip loci not covered by inheritance vectors
    if ivec is None:
        num_skipped += 1
        continue

    candidates = cons.get_candidates(ivec, alleles)
    assign, hap_to_dists = cons.get_best_assign(ivec, alleles, candidates)
    if get_score(hap_to_dists) == 0:
        num_perfect += 1

    num_imperfect, max_error, total_dist = summarize(hap_to_dists)
    summary = Summary(trid, assign, num_imperfect, max_error, total_dist)
    tr_summaries.append((assign, summary))

Skipping ALS1_NIPA1_pathogenic
Skipping BPES_FOXL2_pathogenic
Skipping CANVAS_RFC1_pathogenic
Skipping CCD_RUNX2_pathogenic
Skipping CCHS_PHOX2B_pathogenic


In [ ]:
print(f"Fraction skipped = {num_skipped / num_total}")
print(f"Fraction perfect = {num_perfect / num_total}")

In [ ]:
! ls